In [ ]:
!pip install idx2numpy

In [ ]:
import numpy as np
import pandas as pd
import idx2numpy

import torch
from torch.utils.data import DataLoader, Dataset
import torch.nn as nn
import torch.nn.functional as F
from torch import optim
import torchvision.transforms as transforms

from google.colab import drive
from PIL import Image

#drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
x_test = idx2numpy.convert_from_file('./dataset/t10k-images.idx3-ubyte')
y_test = idx2numpy.convert_from_file('./dataset/t10k-labels.idx1-ubyte')

x_train = idx2numpy.convert_from_file('./dataset/train-images.idx3-ubyte')
y_train = idx2numpy.convert_from_file('./dataset/train-labels.idx1-ubyte')

# converting to torch.tensor
x_test = torch.tensor(x_test, dtype=torch.float32).reshape(x_test.shape[0], -1)
y_test = torch.tensor(y_test).reshape(y_test.shape[0])

x_train = torch.tensor(x_train, dtype=torch.float32).reshape(x_train.shape[0], -1)
y_train = torch.tensor(y_train).reshape(y_train.shape[0])



In [ ]:
class MyData(Dataset):
    def __init__(self, samples='train'):
        self.x_test = x_test
        self.y_test = y_test

        self.x_train = x_train
        self.y_train = y_train

        self.samples = samples

    def __len__(self):
        if self.samples == 'train':
            return self.y_train.shape[0]
        return self.y_test.shape[0]

    def __getitem__(self, index):
        if self.samples == 'train':
            return self.x_train[index], self.y_train[index]
        return self.x_test[index], self.y_test[index]


class MyNet(nn.Module):
    def __init__(self):
        super().__init__()

        self.layer1 = nn.Linear(784, 100)
        self.layer2 = nn.Linear(100, 100)
        self.layer3 = nn.Linear(100, 10)

        self.act1 = nn.ReLU()


    def forward(self, x):
        x = self.act1(self.layer1(x))
        x = self.act1(self.layer2(x))
        return self.layer3(x)



dataset = MyData()
d_train = DataLoader(dataset, batch_size=50, shuffle=True)

model = MyNet()
model.train()

adam = optim.Adam(model.parameters())


In [ ]:
for epoch in range(10):
    for x, y in d_train:
        predict = model(x)
        loss_func = F.cross_entropy(predict, y)

        adam.zero_grad()
        loss_func.backward()
        adam.step()




In [ ]:
model.eval()

dataset = MyData('test')

correct = 0
for x, y in dataset:
    predict = model(x).argmax()
    correct += 1 if predict == y else 0


accuracy = correct / len(dataset)
accuracy

0.9663